# RNN, LSTM and GRU for Sentiment Analysis (PyTorch, IMDB)

**Day 6 of 10 · Deep Learning Architectures Series**

This is the first project on **sequences and text**. I classify IMDB movie reviews as positive or negative with three recurrent architectures, and I test the idea that made LSTM and GRU necessary: **vanilla RNNs struggle with long sequences because gradients vanish through time; gating fixes it.**

**What's inside**
1. Background: RNN, vanishing gradients, LSTM, GRU (with a signal-processing view)
2. Data pipeline with an explicit duplicate/leakage check
3. My own LSTM and GRU **cell** implementations, verified numerically against PyTorch
4. Five experiments: RNN, LSTM, GRU, BiLSTM, BiGRU
5. Results by review length, a **gradient-through-time analysis**, error analysis, and word saliency
6. Analysis, limitations, next steps

> Papers: *Long Short-Term Memory* (Hochreiter and Schmidhuber, 1997) · *Learning Phrase Representations using RNN Encoder-Decoder* (Cho et al., 2014) · *Learning Long-Term Dependencies with Gradient Descent is Difficult* (Bengio et al., 1994) · *Learning Word Vectors for Sentiment Analysis* (Maas et al., 2011, the IMDB dataset)

> ⚙️ **Kaggle setup:** Settings → Accelerator → **GPU** and **Internet → On** (the dataset is downloaded automatically; no "Add Input" is needed). If the download fails, add the Kaggle dataset **"IMDB Dataset of 50K Movie Reviews"** with **+ Add Input** and the notebook will use it instead. Expected runtime: roughly 15 to 30 minutes.

## 1. Background

### Recurrent networks
A recurrent network processes a sequence $x_1, \dots, x_T$ one step at a time, carrying a hidden state $h_t$:

$$h_t = \tanh(W x_t + U h_{t-1} + b)$$

For classification I use the **final hidden state** $h_T$ as a summary of the whole review and feed it to a linear layer.

**Signal-processing view.** A recurrent layer is a **nonlinear IIR filter**: the output depends on its own past through the feedback matrix $U$. If the feedback gain is below 1 the memory decays exponentially, so information from early words fades.

### The vanishing gradient problem
Backpropagation through time multiplies one Jacobian per step:

$$\frac{\partial h_T}{\partial h_t} = \prod_{k=t+1}^{T} \mathrm{diag}\big(1 - h_k^2\big)\, U$$

If the typical gain of these factors is below 1, the gradient from an early word shrinks **exponentially** with the distance $T - t$ (vanishing); if above 1 it explodes. So a vanilla RNN has difficulty learning that a word 300 steps ago matters.

### LSTM
The LSTM adds a **cell state** $c_t$ and three gates (input $i$, forget $f$, output $o$):

$$c_t = f_t \odot c_{t-1} + i_t \odot g_t, \qquad h_t = o_t \odot \tanh(c_t)$$

The update of $c_t$ is **additive**: $\partial c_t / \partial c_{t-1} = f_t$. When the forget gate is close to 1, the gradient flows back almost unchanged (the "constant error carousel"). In filter terms, the forget gate is a **learned, input-dependent decay coefficient**, a time-varying leaky integrator.

### GRU
The GRU merges the cell and hidden state and uses two gates (reset $r$, update $z$):

$$h_t = (1 - z_t) \odot n_t + z_t \odot h_{t-1}, \qquad n_t = \tanh\big(W_n x_t + r_t \odot (U_n h_{t-1})\big)$$

The update gate $z_t$ plays the role of the forget gate: when $z_t \approx 1$ the state is copied forward. It has 3 gates' worth of weights instead of 4, so about 25% fewer recurrent parameters than an LSTM.

### Bidirectional models
A bidirectional RNN runs one RNN forward and one backward over the review and concatenates the two final states. It is a **non-causal** filter, which is fine for offline classification because the whole review is available.

### Experiment design

| ID | Architecture | Direction |
|----|--------------|-----------|
| A | vanilla RNN (tanh) | forward |
| B | LSTM | forward |
| C | GRU | forward |
| D | LSTM | bidirectional |
| E | GRU | bidirectional |

Everything else is identical: tokenization, vocabulary (20,000 words), embedding size 128, hidden size 128 per direction, dropout 0.4 on the final state, Adam (lr 1e-3), gradient clipping at 1.0, batch size 64, 8 epochs, seed 42, and review length capped at 400 tokens. Padding is handled with packed sequences, so the final state is the state after the **last real token**.

In [ ]:
import os, re, copy, time, random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix

SEED = 42
VOCAB_SIZE = 20000     # most frequent training words (plus <pad> and <unk>)
MAX_LEN = 400          # reviews are truncated to their first 400 tokens
EMB_DIM = 128
HIDDEN = 128
DROPOUT = 0.4
BATCH_SIZE = 64
LR = 1e-3
EPOCHS = 8
CLIP = 1.0

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT.mkdir(exist_ok=True)

def save_fig(name):
    plt.savefig(OUT / name, dpi=150, bbox_inches="tight")

print("PyTorch:", torch.__version__, "| Device:", DEVICE)

## 2. Dataset

**IMDB movie reviews** (Maas et al., 2011): 25,000 training and 25,000 test reviews, balanced between positive and negative. The official split uses **different movies** in train and test, which avoids vocabulary that is specific to a movie leaking across the split.

The notebook downloads the official split with the Hugging Face `datasets` library. If that fails, it falls back to a Kaggle CSV (random split, which is *less* strict because reviews of the same movie can land on both sides; the notebook prints a warning in that case).

**Leakage checks.**
- Exact **duplicate reviews** that appear in both train and test (or train and validation) are removed from the evaluation sets.
- The validation set is a random 10% of the official training reviews. Because several reviews of the same movie can end up on both sides of *that* split, validation accuracy may be slightly more optimistic than test accuracy; the official test set is the honest number.

In [ ]:
def load_imdb():
    try:
        from datasets import load_dataset
        try:
            ds = load_dataset("stanfordnlp/imdb")
        except Exception:
            ds = load_dataset("imdb")
        return ("official", list(ds["train"]["text"]), list(ds["train"]["label"]),
                list(ds["test"]["text"]), list(ds["test"]["label"]))
    except Exception as e:
        print("Could not load the dataset from Hugging Face:", repr(e)[:150])
    # Fallback: Kaggle CSV with columns 'review' and 'sentiment'
    for p in sorted(Path("/kaggle/input").rglob("*.csv")):
        try:
            head = pd.read_csv(p, nrows=2)
        except Exception:
            continue
        if {"review", "sentiment"} <= set(head.columns):
            df = pd.read_csv(p).drop_duplicates(subset="review")
            y = (df["sentiment"].str.lower() == "positive").astype(int).values
            tr, te = train_test_split(np.arange(len(df)), test_size=0.2, stratify=y, random_state=SEED)
            texts = df["review"].tolist()
            print(f"WARNING: using {p} with a RANDOM split; reviews of the same movie can appear on both sides, "
                  f"so the scores are optimistic.")
            return "csv", [texts[i] for i in tr], y[tr].tolist(), [texts[i] for i in te], y[te].tolist()
    raise RuntimeError("Dataset not found. Turn Internet ON (Settings) so it can be downloaded, or add the Kaggle dataset "
                       "'IMDB Dataset of 50K Movie Reviews' with '+ Add Input'.")

source, tr_texts_all, tr_labels_all, te_texts, te_labels = load_imdb()
print("data source:", source, "| train+val reviews:", len(tr_texts_all), "| test reviews:", len(te_texts))

# train / validation split
tr_idx, va_idx = train_test_split(np.arange(len(tr_texts_all)), test_size=0.10, stratify=tr_labels_all, random_state=SEED)
train_texts = [tr_texts_all[i] for i in tr_idx]; train_labels = [tr_labels_all[i] for i in tr_idx]
val_texts   = [tr_texts_all[i] for i in va_idx]; val_labels   = [tr_labels_all[i] for i in va_idx]

# leakage check: exact duplicates of training reviews in validation / test
train_set = set(train_texts)
keep_v = [i for i, t in enumerate(val_texts) if t not in train_set]
keep_t = [i for i, t in enumerate(te_texts) if t not in train_set]
print(f"duplicates of training reviews removed: {len(val_texts) - len(keep_v)} from validation, {len(te_texts) - len(keep_t)} from test")
val_texts = [val_texts[i] for i in keep_v]; val_labels = [val_labels[i] for i in keep_v]
te_texts  = [te_texts[i] for i in keep_t];  te_labels  = [te_labels[i] for i in keep_t]
print(f"final sizes: train {len(train_texts):,} | val {len(val_texts):,} | test {len(te_texts):,} | "
      f"positive share: train {np.mean(train_labels):.1%}, test {np.mean(te_labels):.1%}")

In [ ]:
# Tokenization, vocabulary (from the training reviews only), encoding
TOKEN_RE = re.compile(r"[a-z0-9]+(?:'[a-z]+)?|[!?]")
def tokenize(text):
    return TOKEN_RE.findall(text.lower().replace("<br />", " "))

counter = Counter(tok for t in train_texts for tok in tokenize(t))
itos = ["<pad>", "<unk>"] + [w for w, c in counter.most_common(VOCAB_SIZE) if c >= 2]
stoi = {w: i for i, w in enumerate(itos)}
print("vocabulary size:", len(itos), "| most frequent:", itos[2:12])

def encode(texts):
    n = len(texts)
    X = np.zeros((n, MAX_LEN), np.int64); L = np.zeros(n, np.int64); O = np.zeros(n, np.int64)
    for i, t in enumerate(texts):
        ids = [stoi.get(w, 1) for w in tokenize(t)] or [1]
        O[i] = len(ids)
        ids = ids[:MAX_LEN]; L[i] = len(ids); X[i, :L[i]] = ids
    return X, L, O

Xtr_np, Ltr_np, Otr_np = encode(train_texts)
Xva_np, Lva_np, Ova_np = encode(val_texts)
Xte_np, Lte_np, Ote_np = encode(te_texts)
ytr_np, yva_np, yte_np = np.array(train_labels), np.array(val_labels), np.array(te_labels)

unk_rate = (Xva_np == 1).sum() / Lva_np.sum()
print(f"unknown-token rate on validation reviews: {unk_rate:.2%} | reviews longer than {MAX_LEN} tokens (truncated): "
      f"{(Otr_np > MAX_LEN).mean():.1%} of train, {(Ote_np > MAX_LEN).mean():.1%} of test")

to_t = lambda a, dt=None: torch.as_tensor(a, device=DEVICE, dtype=dt)
Xtr, Ltr, ytr = to_t(Xtr_np), to_t(Ltr_np), to_t(ytr_np, torch.float32)
Xva, Lva, yva = to_t(Xva_np), to_t(Lva_np), to_t(yva_np, torch.float32)
Xte, Lte, yte = to_t(Xte_np), to_t(Lte_np), to_t(yte_np, torch.float32)

plt.figure(figsize=(7, 3.5))
plt.hist(np.clip(Otr_np, 0, 1200), bins=60, color="tab:blue")
plt.axvline(MAX_LEN, color="red", ls="--", label=f"max length = {MAX_LEN}")
plt.xlabel("review length (tokens, clipped at 1200)"); plt.ylabel("reviews"); plt.legend()
plt.title("Review length distribution (training set)")
plt.tight_layout()
save_fig("length_distribution.png")
plt.show()

## 3. LSTM and GRU Cells, Implemented by Hand

To make sure I understand the gate equations, I implement an **LSTM cell** and a **GRU cell** myself and check them numerically against `nn.LSTMCell` and `nn.GRUCell` by copying the same weights. The main experiments then use PyTorch's optimized (cuDNN) `nn.RNN`, `nn.LSTM` and `nn.GRU`, which compute the same equations much faster.

PyTorch's gate order is **(i, f, g, o)** for the LSTM and **(r, z, n)** for the GRU. Note that in the GRU the reset gate multiplies the *hidden projection* of the candidate, including its bias.

In [ ]:
class MyLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        k = 1.0 / hidden_size ** 0.5
        self.W_ih = nn.Parameter(torch.empty(4 * hidden_size, input_size).uniform_(-k, k))
        self.W_hh = nn.Parameter(torch.empty(4 * hidden_size, hidden_size).uniform_(-k, k))
        self.b_ih = nn.Parameter(torch.empty(4 * hidden_size).uniform_(-k, k))
        self.b_hh = nn.Parameter(torch.empty(4 * hidden_size).uniform_(-k, k))

    def forward(self, x, state):
        h, c = state
        gates = x @ self.W_ih.T + self.b_ih + h @ self.W_hh.T + self.b_hh
        i, f, g, o = gates.chunk(4, dim=1)
        i, f, o, g = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o), torch.tanh(g)
        c_new = f * c + i * g                  # additive cell update
        h_new = o * torch.tanh(c_new)
        return h_new, c_new


class MyGRUCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        k = 1.0 / hidden_size ** 0.5
        self.W_ih = nn.Parameter(torch.empty(3 * hidden_size, input_size).uniform_(-k, k))
        self.W_hh = nn.Parameter(torch.empty(3 * hidden_size, hidden_size).uniform_(-k, k))
        self.b_ih = nn.Parameter(torch.empty(3 * hidden_size).uniform_(-k, k))
        self.b_hh = nn.Parameter(torch.empty(3 * hidden_size).uniform_(-k, k))

    def forward(self, x, h):
        gi = x @ self.W_ih.T + self.b_ih
        gh = h @ self.W_hh.T + self.b_hh
        i_r, i_z, i_n = gi.chunk(3, dim=1)
        h_r, h_z, h_n = gh.chunk(3, dim=1)
        r = torch.sigmoid(i_r + h_r)
        z = torch.sigmoid(i_z + h_z)
        n = torch.tanh(i_n + r * h_n)
        return (1 - z) * n + z * h             # leaky integration of the previous state


# Numerical check against PyTorch's reference cells (same weights, same inputs)
torch.manual_seed(0)
I, H, B = 16, 32, 5
x, h, c = torch.randn(B, I), torch.randn(B, H), torch.randn(B, H)

ref, mine = nn.LSTMCell(I, H), MyLSTMCell(I, H)
with torch.no_grad():
    mine.W_ih.copy_(ref.weight_ih); mine.W_hh.copy_(ref.weight_hh); mine.b_ih.copy_(ref.bias_ih); mine.b_hh.copy_(ref.bias_hh)
(h1, c1), (h2, c2) = ref(x, (h, c)), mine(x, (h, c))
d_lstm = max((h1 - h2).abs().max().item(), (c1 - c2).abs().max().item())

ref, mine = nn.GRUCell(I, H), MyGRUCell(I, H)
with torch.no_grad():
    mine.W_ih.copy_(ref.weight_ih); mine.W_hh.copy_(ref.weight_hh); mine.b_ih.copy_(ref.bias_ih); mine.b_hh.copy_(ref.bias_hh)
d_gru = (ref(x, h) - mine(x, h)).abs().max().item()

print(f"LSTM cell: max abs difference vs nn.LSTMCell = {d_lstm:.2e} ->", "matches" if d_lstm < 1e-5 else "MISMATCH, check the gate order")
print(f"GRU  cell: max abs difference vs nn.GRUCell  = {d_gru:.2e} ->", "matches" if d_gru < 1e-5 else "MISMATCH, check the equations")

## 4. Model

`SentimentRNN` = embedding (learned from scratch, 128 dimensions) → recurrent layer (`RNN`, `LSTM` or `GRU`, optionally bidirectional) → dropout → linear layer producing **one logit** (positive vs negative). Sequences are **packed**, so the recurrent layer never processes padding and the final state is the state after the last real token.

| Model | Recurrent parameters | Total parameters |
|-------|---------------------|------------------|
| RNN | 33,024 | 2,593,409 |
| LSTM | 132,096 | 2,692,481 |
| GRU | 99,072 | 2,659,457 |
| BiLSTM | 264,192 | 2,824,705 |
| BiGRU | 198,144 | 2,758,657 |

The embedding table (20,002 × 128 = 2,560,256 parameters) dominates the parameter count in every model.

In [ ]:
class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, rnn_type="LSTM", emb_dim=EMB_DIM, hidden=HIDDEN, bidirectional=False, dropout=DROPOUT):
        super().__init__()
        self.rnn_type, self.bi = rnn_type, bidirectional
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        cls = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[rnn_type]
        extra = {"nonlinearity": "tanh"} if rnn_type == "RNN" else {}
        self.rnn = cls(emb_dim, hidden, batch_first=True, bidirectional=bidirectional, **extra)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden * (2 if bidirectional else 1), 1)

    def forward(self, x, lengths, return_emb=False):
        emb = self.emb(x)
        if return_emb:
            emb.retain_grad()                  # used for gradient / saliency analysis
        packed = nn.utils.rnn.pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h = self.rnn(packed)
        if self.rnn_type == "LSTM":
            h = h[0]                           # (h_n, c_n) -> h_n
        h = torch.cat([h[-2], h[-1]], dim=1) if self.bi else h[-1]     # final state(s) after the last real token
        logit = self.fc(self.dropout(h)).squeeze(1)
        return (logit, emb) if return_emb else logit

def count_params(m):
    return sum(p.numel() for p in m.parameters())

for name, t, bi in [("RNN", "RNN", False), ("LSTM", "LSTM", False), ("GRU", "GRU", False),
                    ("BiLSTM", "LSTM", True), ("BiGRU", "GRU", True)]:
    print(f"{name:7s} {count_params(SentimentRNN(len(itos), t, bidirectional=bi)):,} parameters")

## 5. Training Setup

- **Loss:** binary cross-entropy with logits | **Optimizer:** Adam (lr 1e-3) | **Gradient clipping:** norm 1.0 (the standard remedy for *exploding* gradients; vanishing gradients are not fixed by clipping)
- **Epochs:** 8 | **Batch size:** 64 | whole dataset kept on the GPU (no DataLoader workers)
- **Model selection:** best validation accuracy; the test set is evaluated once per run
- Same seed and splits for all runs

In [ ]:
@torch.no_grad()
def predict(model, X, L, bs=256):
    model.eval()
    out = []
    for i in range(0, len(X), bs):
        out.append(torch.sigmoid(model(X[i:i + bs], L[i:i + bs])).cpu())
    return torch.cat(out).numpy()

def run_epoch(model, X, L, y, optimizer=None):
    training = optimizer is not None
    model.train(training)
    idx = torch.randperm(len(X), device=DEVICE) if training else torch.arange(len(X), device=DEVICE)
    total_loss, correct = 0.0, 0
    with torch.set_grad_enabled(training):
        for i in range(0, len(idx), BATCH_SIZE):
            j = idx[i:i + BATCH_SIZE]
            logits = model(X[j], L[j])
            loss = F.binary_cross_entropy_with_logits(logits, y[j])
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), CLIP)
                optimizer.step()
            total_loss += loss.item() * len(j)
            correct += ((logits > 0).float() == y[j]).sum().item()
    return total_loss / len(X), correct / len(X)

def train_experiment(name, rnn_type, bidirectional):
    print(f"\n=== {name} | {rnn_type} | bidirectional={bidirectional} ===")
    torch.manual_seed(SEED)
    model = SentimentRNN(len(itos), rnn_type, bidirectional=bidirectional).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val, best_epoch, best_state = -1.0, 0, None
    t0 = time.time()
    for ep in range(1, EPOCHS + 1):
        tl, ta = run_epoch(model, Xtr, Ltr, ytr, optimizer)
        vl, va = run_epoch(model, Xva, Lva, yva)
        for k, v in zip(hist, [tl, vl, ta, va]):
            hist[k].append(v)
        if va > best_val:
            best_val, best_epoch = va, ep
            best_state = copy.deepcopy(model.state_dict())
        print(f"Epoch {ep}/{EPOCHS} | train loss {tl:.3f} acc {ta:.4f} | val loss {vl:.3f} acc {va:.4f} | {time.time() - t0:.0f}s")
    elapsed = time.time() - t0

    model.load_state_dict(best_state)
    probs = predict(model, Xte, Lte)
    pred = (probs > 0.5).astype(int)
    test_acc = float((pred == yte_np).mean()); test_f1 = f1_score(yte_np, pred)
    print(f"-> best epoch {best_epoch} | val acc {best_val:.4f} | test acc {test_acc:.4f} | F1 {test_f1:.4f} | {elapsed:.0f}s")
    return dict(model=model, hist=hist, rnn_type=rnn_type, bi=bidirectional, params=count_params(model),
                best_epoch=best_epoch, best_val_acc=best_val, test_acc=test_acc, test_f1=test_f1,
                test_probs=probs, time=elapsed)

results = {}

### Experiment A: RNN

In [ ]:
results["A: RNN"] = train_experiment("A: RNN", "RNN", False)

### Experiment B: LSTM

In [ ]:
results["B: LSTM"] = train_experiment("B: LSTM", "LSTM", False)

### Experiment C: GRU

In [ ]:
results["C: GRU"] = train_experiment("C: GRU", "GRU", False)

### Experiment D: BiLSTM

In [ ]:
results["D: BiLSTM"] = train_experiment("D: BiLSTM", "LSTM", True)

### Experiment E: BiGRU

In [ ]:
results["E: BiGRU"] = train_experiment("E: BiGRU", "GRU", True)

## 6. Results

In [ ]:
rows = []
for name, r in results.items():
    rows.append({
        "Config": name,
        "Params": f"{r['params']:,}",
        "Best epoch": r["best_epoch"],
        "Val acc (%)": round(r["best_val_acc"] * 100, 2),
        "Test acc (%)": round(r["test_acc"] * 100, 2),
        "Test F1 (%)": round(r["test_f1"] * 100, 2),
        "Time (s)": round(r["time"]),
    })
summary = pd.DataFrame(rows)
summary.to_csv(OUT / "results_summary.csv", index=False)
summary

In [ ]:
# Learning curves
names = list(results)
fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
for n in names:
    h = results[n]["hist"]; ep = np.arange(1, EPOCHS + 1)
    ax[0].plot(ep, np.array(h["val_acc"]) * 100, marker="o", ms=3, label=n)
    ax[1].plot(ep, h["train_loss"], marker="o", ms=3, label=n)
    ax[2].plot(ep, h["val_loss"], marker="o", ms=3, label=n)
ax[0].set_title("Validation accuracy (%)"); ax[1].set_title("Training loss"); ax[2].set_title("Validation loss")
for a in ax:
    a.set_xlabel("epoch"); a.grid(alpha=0.3)
ax[0].legend(fontsize=8)
plt.tight_layout()
save_fig("training_curves_all.png")
plt.show()

In [ ]:
# Test accuracy overall and by review length (length = original number of tokens, before truncation)
bins = [(0, 100), (100, 200), (200, 400), (400, 10**9)]
labels_b = ["< 100", "100-199", "200-399", ">= 400 (truncated)"]
fig, ax = plt.subplots(1, 2, figsize=(16, 4.5), gridspec_kw={"width_ratios": [1, 1.6]})

accs = [results[n]["test_acc"] * 100 for n in names]
bars = ax[0].bar([n.split(":")[0] for n in names], accs, color=["tab:gray", "tab:blue", "tab:green", "tab:purple", "tab:red"])
for b, a_ in zip(bars, accs):
    ax[0].text(b.get_x() + b.get_width() / 2, a_ + 0.3, f"{a_:.2f}", ha="center", fontsize=9)
ax[0].set_ylim(max(40, min(accs) - 5), min(100, max(accs) + 3)); ax[0].set_title("Test accuracy (%)")
ax[0].set_xlabel("A: RNN | B: LSTM | C: GRU | D: BiLSTM | E: BiGRU", fontsize=8)

w = 0.16
for k, n in enumerate(names):
    pred = (results[n]["test_probs"] > 0.5).astype(int)
    vals = [((pred == yte_np)[(Ote_np >= lo) & (Ote_np < hi)]).mean() * 100 for lo, hi in bins]
    ax[1].bar(np.arange(len(bins)) + (k - 2) * w, vals, w, label=n)
ax[1].set_xticks(np.arange(len(bins))); ax[1].set_xticklabels(labels_b)
ax[1].set_ylim(max(40, min(accs) - 15), 100); ax[1].set_ylabel("test accuracy (%)")
ax[1].set_title("Test accuracy by review length (tokens)"); ax[1].legend(fontsize=8, ncol=2)
plt.tight_layout()
save_fig("accuracy_by_length.png")
plt.show()

print("reviews per length bucket:", {l: int(((Ote_np >= lo) & (Ote_np < hi)).sum()) for l, (lo, hi) in zip(labels_b, bins)})

## 7. Do Gradients Vanish Through Time?

To test the central claim directly, I measure how strongly the model output depends on each position of a long review. For the trained forward models (RNN, LSTM, GRU) and for 256 test reviews that are at least 400 tokens long (so all positions line up), I compute the **norm of the gradient of the output logit with respect to the embedding of the word at each position**. A curve that falls steeply toward the start of the review means that early words have almost no influence on the prediction (vanishing gradient through time).

The curves are normalized by their average over the last 10 tokens, and the x-axis shows the distance from the **end** of the review. (cuDNN is disabled for this computation because its RNN backward pass requires training mode.)

In [ ]:
def grad_profile(model, X, L):
    model.eval()
    model.zero_grad(set_to_none=True)
    with torch.backends.cudnn.flags(enabled=False):
        logit, emb = model(X, L, return_emb=True)
        logit.sum().backward()
    return emb.grad.norm(dim=-1).mean(0).cpu().numpy()          # mean over reviews, per position

long_ids = np.where(Ote_np >= MAX_LEN)[0][:256]
print(f"{len(long_ids)} long test reviews used")
long_t = torch.as_tensor(long_ids, device=DEVICE)
Xl, Ll = Xte[long_t], Lte[long_t]

plt.figure(figsize=(8.5, 4.5))
ratios = {}
for n in ["A: RNN", "B: LSTM", "C: GRU"]:
    g = grad_profile(results[n]["model"], Xl, Ll) + 1e-12
    g = g / g[-10:].mean()
    plt.semilogy(np.arange(MAX_LEN)[::-1], g, label=n)
    ratios[n] = g[:50].mean() / g[-50:].mean()
plt.xlabel("distance from the end of the review (tokens)"); plt.ylabel("gradient norm, relative to the last 10 tokens (log scale)")
plt.title("Influence of each word on the output (trained models)")
plt.legend(); plt.grid(alpha=0.3)
save_fig("gradient_through_time.png")
plt.show()

for n, r_ in ratios.items():
    print(f"{n}: gradient at the first 50 tokens / last 50 tokens = {r_:.4f}")

## Error Analysis (best model)

The best model is selected by **validation** accuracy.

In [ ]:
best_name = max(results, key=lambda k: results[k]["best_val_acc"])
best = results[best_name]
probs = best["test_probs"]; pred = (probs > 0.5).astype(int)
print("Best config by validation accuracy:", best_name, f"| test acc {best['test_acc']*100:.2f}%")

cm = confusion_matrix(yte_np, pred)
tn, fp, fn, tp = cm.ravel()
print(f"confusion matrix [[TN FP] [FN TP]]: {cm.tolist()} | precision {tp / (tp + fp):.3f} | recall {tp / (tp + fn):.3f}")

# Confidence of correct vs wrong predictions
conf = np.where(pred == 1, probs, 1 - probs)
print(f"mean confidence: correct {conf[pred == yte_np].mean():.3f} | wrong {conf[pred != yte_np].mean():.3f}")

# Most confident mistakes
wrong = np.where(pred != yte_np)[0]
worst = wrong[np.argsort(-conf[wrong])][:5]
for i in worst:
    t = te_texts[i].replace("<br />", " ")
    print(f"\n[true: {'positive' if yte_np[i] else 'negative'} | predicted: {'positive' if pred[i] else 'negative'} "
          f"| confidence {conf[i]:.3f} | {Ote_np[i]} tokens]\n  {t[:350]}...")

### Which words drive a prediction? (gradient saliency)

For a few test reviews I compute the norm of the gradient of the output logit with respect to each word's embedding and list the most influential words. This is a rough attribution method (gradients are local), so I treat it as an illustration, not as proof of what the model "understands".

In [ ]:
def saliency(model, i):
    model.eval(); model.zero_grad(set_to_none=True)
    with torch.backends.cudnn.flags(enabled=False):
        logit, emb = model(Xte[i:i + 1], Lte[i:i + 1], return_emb=True)
        logit.sum().backward()
    n = int(Lte_np[i])
    return emb.grad.norm(dim=-1)[0, :n].cpu().numpy(), torch.sigmoid(logit).item()

rs = np.random.RandomState(SEED)
correct_pos = rs.choice(np.where((pred == yte_np) & (yte_np == 1))[0], 2, replace=False)
correct_neg = rs.choice(np.where((pred == yte_np) & (yte_np == 0))[0], 2, replace=False)
examples = list(correct_pos) + list(correct_neg) + list(worst[:2])

for i in examples:
    s, p_pos = saliency(best["model"], i)
    ids = Xte_np[i, :len(s)]
    order = np.argsort(-s)[:10]
    top = ", ".join(f"{itos[ids[k]]}({s[k] / s.max():.2f})" for k in order)
    status = "correct" if pred[i] == yte_np[i] else "WRONG"
    print(f"[true {'pos' if yte_np[i] else 'neg'} | P(pos) = {p_pos:.2f} | {status}]  top words: {top}")

In [ ]:
# Save the best model and the vocabulary
import json
torch.save(best["model"].state_dict(), OUT / "sentiment_rnn_best.pth")
(OUT / "vocab.json").write_text(json.dumps(itos))
print("Saved to", OUT)

## 8. Analysis

### Key findings

**1. Gated models clearly beat the vanilla RNN.** Test accuracy: RNN 79.48%, LSTM 85.42% (+5.9 pp), GRU 87.84% (+8.4 pp). The RNN also trains poorly: its training loss only falls from 0.66 to 0.28 in 8 epochs (the gated models reach 0.02 to 0.10), its validation accuracy is erratic (it drops from 76% to 68% at epoch 4), and its best epoch is the last one, so it is still improving. Part of the gap is therefore slower optimization within an 8-epoch budget, not only a lower final ceiling.

**2. Gradients vanish through time in the RNN, exactly as the theory says.** For long reviews, the gradient of the output with respect to a word's embedding, relative to the last 10 tokens:
- *RNN:* similar to the other models for the last 50 tokens, then it collapses: roughly 10 orders of magnitude lower at 150 to 200 tokens from the end, and effectively zero beyond about 250 tokens (the flat line at the bottom is my numerical floor, not a measured value). The ratio of the gradient at the first 50 tokens to the last 50 tokens is printed as 0.0000.
- *LSTM:* decays gradually, about 2 orders of magnitude over 400 tokens (ratio 0.015).
- *GRU:* almost flat, within about a factor of 2 to 5 across the whole review (ratio 0.54), with a slight rise at the very beginning of the review.

So the ranking of gradient flow (GRU > LSTM >> RNN) matches the ranking in accuracy on long reviews. Caveats: gradient magnitude shows how much the output *can be influenced* by a position, not that the model uses it well; the models are evaluated at their own best epochs (5 to 8) after different amounts of training; and this is one seed.

**3. Accuracy drops with review length for every model, most for the RNN.** Reading the grouped bar chart (approximate values): from reviews under 100 tokens to reviews of 400 tokens or more, the RNN falls from about 84% to 72%, the LSTM from 88% to 78%, the BiLSTM from 88% to 76%, the BiGRU from 89% to 81% and the GRU from 89% to 83%. The largest step for all models is the last bucket (3,154 of 24,893 test reviews, about 13%), which is exactly where reviews are **truncated to their first 400 tokens**. A plausible reason (a hypothesis) is that the verdict of a review often comes at its end and is cut off, on top of the long-range difficulty. The GRU is the most robust to length.

**4. GRU beat LSTM under identical settings.** GRU 87.84% vs LSTM 85.42% test accuracy (+2.4 pp), F1 87.98% vs 85.68%, validation 87.86% vs 85.73%, with 25% fewer recurrent parameters (99,072 vs 132,096) and the same training time (57 s). The LSTM also trains more slowly (final training loss 0.10 vs 0.03). A plausible explanation (hypothesis, not tested) is that the LSTM needs more epochs or a different setup, for example initializing the forget-gate bias to 1, to match the GRU. I would not conclude that GRU is better than LSTM in general from one seed.

**5. Bidirectional models did not help here.** BiLSTM 85.31% vs LSTM 85.42%, BiGRU 87.07% vs GRU 87.84%, while costing about 60% more training time (93 s vs 57 s). The differences are within about one point, which is the size of the gap between validation and test accuracy for some runs (BiLSTM: 86.81% vs 85.31%), so I treat bidirectionality as showing no benefit rather than harm. Possible reasons (hypotheses): the bidirectional models overfit faster (BiGRU reaches a training loss of 0.02 and its validation loss starts rising after epoch 3), and with the last hidden state as a summary the backward pass ends at the *beginning* of the review.

**6. Strong overfitting, and the models become overconfident.** For all gated models the validation loss reaches its minimum at epochs 3 to 6 and then rises (for example BiGRU from about 0.33 to 0.62), while the training loss keeps falling toward 0.02 to 0.1. Validation accuracy stays on a plateau of 86% to 88%, so the models become more confident without becoming more accurate. About 98% of the parameters are in the embedding table (2.56M of 2.6M to 2.8M), which makes memorization easy. Selecting the epoch by validation accuracy (as I did) picks a later, more overconfident epoch than selecting by validation loss would. Mean confidence on the GRU's mistakes is still 0.872 (vs 0.968 on correct predictions), so confidence is only a weak signal for flagging errors.

**7. Error analysis (GRU, the best model on validation).** The confusion matrix is [[10,791 TN, 1,626 FP], [1,401 FN, 11,075 TP]] (precision 0.872, recall 0.888); it predicts "positive" slightly more often than the true rate (51.0% vs 50.1%). All five of the most confident mistakes have confidence 1.000. From the visible first part of those reviews, they look like mixed or contradictory reviews (praise for some aspects and criticism of others), sarcasm or irony, a review that calls the film "boring" yet is labeled positive, and one 999-token review whose verdict is probably cut off by the truncation. At least some may also be ambiguous or noisy labels. I only saw the first 350 characters of each, so these are impressions, not a measured breakdown.

**8. Saliency suggests partly shortcut-like cues.** Gradient saliency (a rough, local attribution) gives some sensible top words (for example "blessed", "favorite", "bad", "avoid", "ridiculous", "recommended", "highly"), but also many arbitrary ones ("be", "me", "movies", "at"). Two patterns stand out: the number tokens "5", "10", "7", "8" (likely rating mentions) get the highest score in three examples, including both confident mistakes, and the names "chuck" and "norris" are among the top words for a negative review. This suggests the model leans on rating numbers and on entity names, which can be misleading cues (I did not see the surrounding context of those numbers, so this remains a hypothesis).

**9. Cost.** All runs take about 1 to 1.5 minutes, so running several seeds per configuration is cheap and would settle the questions above.

### Limitations
- Single run per configuration (one seed). Differences of about 1 pp or less (LSTM vs BiLSTM, GRU vs BiGRU) are not conclusive; the +2.4 pp GRU vs LSTM gap and the RNN gap are larger, but still come from one seed.
- Reviews are truncated to 400 tokens and the embedding is learned from scratch for only 8 epochs; the RNN is clearly undertrained.
- About 107 test reviews (derived from the test size of 24,893 vs the official 25,000) were removed as exact duplicates of training reviews; near-duplicates were not checked. The validation split shares movies with the training split.
- Gradient and saliency analyses are local and approximate, and the confident-mistake review is based on the first 350 characters only.
- The setup is not state of the art: no pretrained embeddings, no attention, no hyperparameter search.

### Possible improvements
- Run 5 seeds per configuration and report mean ± std; select the epoch by validation loss as well as accuracy
- Initialize the LSTM forget-gate bias to 1; tune hidden size, dropout, learning rate; add embedding dropout or weight decay
- Pretrained embeddings (GloVe, fastText); attention or max/mean pooling instead of the last hidden state; keep the last 400 tokens or both ends of long reviews instead of only the first 400
- Test the rating-number shortcut (mask or randomize numbers) and calibrate the output probabilities
- Compare with a fine-tuned Transformer on the same split (Day 8)

### Next in this series
**Day 7:** Seq2Seq with attention for machine translation.

### References
- Hochreiter, S., Schmidhuber, J. (1997). *Long Short-Term Memory.* Neural Computation.
- Cho, K. et al. (2014). *Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation.* EMNLP.
- Bengio, Y., Simard, P., Frasconi, P. (1994). *Learning long-term dependencies with gradient descent is difficult.* IEEE Transactions on Neural Networks.
- Pascanu, R., Mikolov, T., Bengio, Y. (2013). *On the difficulty of training recurrent neural networks.* ICML.
- Maas, A. et al. (2011). *Learning Word Vectors for Sentiment Analysis.* ACL.